# Network Service Availability Prediction
### Case Study 29 – Anomaly Classification on Telecom KPI Time Series

## 1. Project Introduction

**Problem statement.** Organizations depend on network services for communication and business operations. Degradation and anomalies in the network can reduce service reliability. This project analyzes network telemetry (KPI time series), identifies abnormal behaviour, and builds a machine learning model to estimate **network anomaly risk**.

**Objectives**
1. Load and clean the nested `telecom_ts_dataset`.
2. Explore network behaviour and the anomaly class.
3. Turn KPI time series into simple numerical features.
4. Train and compare Dummy, Logistic Regression and Random Forest models.
5. Evaluate with Accuracy, Precision, Recall, F1-score and False Alarm Rate.

**Research questions**
- Can KPI summary features separate anomalous records from normal ones?
- Which models detect anomalies best while keeping false alarms low?
- Are the timestamps trustworthy enough for a time-based split?

**Dataset overview.** 32,000 records with 9 original columns: `start_time`, `end_time`, `sampling_rate`, `KPIs`, `description`, `anomalies`, `statistics`, `labels`, `QnA`. Nested columns are stored as strings and must be parsed. The target `labels['anomaly_present']` is imbalanced (about 3.86% `Yes`).

**Scope and limitations (important)**
- The target `anomaly_present` means *an anomaly is present in the trace*. It is **not** a confirmed service outage.
- This notebook performs **retrospective anomaly classification**, not prospective outage forecasting.
- Section 8 checks whether the data contains a real outage label. No outage labels will be invented.

## 2. Library Setup
Pandas/NumPy for data, Matplotlib/Seaborn for plots, Scikit-learn for modelling. XGBoost is optional and not used, to keep the notebook simple.

In [ ]:
import ast
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report,
                             precision_recall_curve, average_precision_score,
                             ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42
print("Libraries loaded.")

## 3. Google Drive Data Loading
Change `DATA_PATH` to where `telecom_ts_dataset` is stored in your Drive (`.csv` or `.parquet`). The cell checks that the file exists and prints the files in the folder if it does not.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# >>> EDIT THIS PATH <<<
DATA_PATH = "/content/drive/MyDrive/telecom_ts_dataset.csv"

if not os.path.exists(DATA_PATH):
    folder = os.path.dirname(DATA_PATH)
    msg = f"File not found: {DATA_PATH}\nCheck the file name and folder in your Google Drive."
    if os.path.isdir(folder):
        msg += f"\nFiles in '{folder}': {os.listdir(folder)[:20]}"
    else:
        msg += f"\nFolder '{folder}' does not exist."
    raise FileNotFoundError(msg)

ext = os.path.splitext(DATA_PATH)[1].lower()
if ext == ".csv":
    df = pd.read_csv(DATA_PATH)
elif ext == ".parquet":
    df = pd.read_parquet(DATA_PATH)
else:
    raise ValueError(f"Unsupported file type '{ext}'. Use .csv or .parquet.")

print("Dataset shape:", df.shape)
df.head(3)

## 4. Data Understanding
First we check that all nine expected columns exist. Then we parse the nested columns (`KPIs`, `statistics`, `anomalies`, `labels`) from strings into Python objects. `QnA` and `description` are text metadata that will not be used as model inputs, so they are left unparsed.

The parser handles strings like `{'kpi': array([1.0, 2.0], dtype=float32)}`. It first tries the safe `ast.literal_eval`, then a restricted `eval` that only knows `array`, `nan`, `inf` and NumPy dtypes (your own Drive file, so this is acceptable).

In [ ]:
EXPECTED = ["start_time", "end_time", "sampling_rate", "KPIs", "description",
            "anomalies", "statistics", "labels", "QnA"]
missing_cols = [c for c in EXPECTED if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing expected columns: {missing_cols}. Found: {list(df.columns)}")

print("Shape:", df.shape)
print(df.dtypes)

SAFE_NAMES = {"array": np.array, "np": np, "nan": np.nan, "inf": np.inf,
              "float32": np.float32, "float64": np.float64,
              "int32": np.int32, "int64": np.int64}

def parse_cell(x):
    """Turn a string such as "{'a': array([1,2])}" into a Python object. Returns None on failure."""
    if not isinstance(x, str):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        pass
    try:
        return eval(x, {"__builtins__": {}}, SAFE_NAMES)
    except Exception:
        return None

for col in ["KPIs", "statistics", "anomalies", "labels"]:
    df[col + "_parsed"] = df[col].apply(parse_cell)
    failed = df[col + "_parsed"].apply(lambda v: v is None).sum()
    print(f"{col}: {failed} records could not be parsed")

In [ ]:
def first_valid(col, kind=dict):
    """Return the first parsed object of the given type in a column."""
    for v in df[col]:
        if isinstance(v, kind):
            return v
    raise TypeError(f"No valid {kind.__name__} found in column '{col}'.")

kpis0 = first_valid("KPIs_parsed")
print("KPI names:", list(kpis0.keys()))
for k, v in kpis0.items():
    print(f"  {k}: type={type(v).__name__}, shape={np.asarray(v).shape}")

stats0 = first_valid("statistics_parsed")
print("\nstatistics keys (first 10):", list(stats0.keys())[:10])

print("\nanomalies example (type, first 300 chars):", type(df['anomalies_parsed'].iloc[0]).__name__,
      repr(df['anomalies_parsed'].iloc[0])[:300])

labels0 = first_valid("labels_parsed")
print("\nlabels keys:", list(labels0.keys()))
print("labels example:", labels0)

In [ ]:
# Extract the contextual labels and the target into flat columns
LABEL_COLS = ["zone", "application", "mobility", "congestion", "anomaly_present"]
for c in LABEL_COLS:
    df[c] = df["labels_parsed"].apply(lambda d: d.get(c) if isinstance(d, dict) else None)

print("Missing label values:\n", df[LABEL_COLS].isna().sum(), "\n")
for c in LABEL_COLS:
    print(f"--- {c} ---")
    print(df[c].value_counts(dropna=False), "\n")

In [ ]:
# Missing values, duplicates, timestamps, sampling rate
print("Missing values in original columns:\n", df[EXPECTED].isna().sum(), "\n")
n_dup = df[EXPECTED].astype(str).duplicated().sum()
print("Exact duplicate records:", n_dup)

start = pd.to_datetime(df["start_time"], errors="coerce", utc=True)
end = pd.to_datetime(df["end_time"], errors="coerce", utc=True)
print("\nInvalid start_time:", start.isna().sum(), "| Invalid end_time:", end.isna().sum())

duration = (end - start).dt.total_seconds()
print("Trace duration (seconds):")
print(duration.describe())
print("Records with duration <= 0:", (duration <= 0).sum())

print("\nRecords per year:")
print(start.dt.year.value_counts().sort_index())

now = pd.Timestamp.now(tz="UTC")
print("\nRecords with start_time later than today:", (start > now).sum())

# Largest gaps between consecutive (sorted) start times
sorted_start = start.dropna().sort_values()
gap_df = pd.DataFrame({"start_time": sorted_start, "gap_days": sorted_start.diff().dt.days})
print("\nFive largest gaps between consecutive records:")
display(gap_df.nlargest(5, "gap_days"))

print("sampling_rate values:")
print(df["sampling_rate"].value_counts().head(10))

## 5. Data Cleaning
- Convert timestamps to datetime (invalid ones become `NaT`, not deleted).
- Remove exact duplicate records **only** if they exist (an exact copy adds no information).
- Validate the numeric KPI arrays. Records with problems are **kept and counted**; bad values become missing and are imputed later using training data only.

In [ ]:
df["start_dt"] = pd.to_datetime(df["start_time"], errors="coerce", utc=True)
df["end_dt"] = pd.to_datetime(df["end_time"], errors="coerce", utc=True)
df["duration_sec"] = (df["end_dt"] - df["start_dt"]).dt.total_seconds()

dup_mask = df[EXPECTED].astype(str).duplicated()
if dup_mask.sum() > 0:
    print(f"Removing {dup_mask.sum()} exact duplicate records (identical in all 9 original columns).")
    df = df[~dup_mask].reset_index(drop=True)
else:
    print("No exact duplicates found - nothing removed.")
print("Shape after cleaning step:", df.shape)

In [ ]:
def to_array(v):
    """Convert a KPI value to a 1-D float array. Returns None if impossible."""
    try:
        a = np.asarray(v, dtype=float).ravel()
        return a if a.size > 0 else None
    except Exception:
        return None

kpi_names = list(first_valid("KPIs_parsed").keys())
problems = {"record_not_dict": 0, "kpi_missing": 0, "kpi_not_numeric": 0, "kpi_has_nan": 0, "kpi_has_inf": 0}
trace_len = []
for d in df["KPIs_parsed"]:
    if not isinstance(d, dict):
        problems["record_not_dict"] += 1
        trace_len.append(np.nan)
        continue
    first_len = np.nan
    for k in kpi_names:
        if k not in d:
            problems["kpi_missing"] += 1
            continue
        a = to_array(d[k])
        if a is None:
            problems["kpi_not_numeric"] += 1
            continue
        if np.isnan(a).any(): problems["kpi_has_nan"] += 1
        if np.isinf(a).any(): problems["kpi_has_inf"] += 1
        if np.isnan(first_len): first_len = len(a)
    trace_len.append(first_len)

print("Problem counts (records, or record-KPI pairs for the KPI items):")
for k, v in problems.items():
    print(f"  {k}: {v}")
df["trace_len"] = trace_len
print("\nTrace length (number of samples per KPI):")
print(df["trace_len"].describe())
print("\nNo records were deleted for these issues; invalid values become NaN and are imputed later.")

## 6. Exploratory Data Analysis

In [ ]:
# Temporary 0/1 target just for plotting (the official target is built in Section 8)
eda_target = df["anomaly_present"].map({"No": 0, "Yes": 1})
year = df["start_dt"].dt.year

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

df["anomaly_present"].value_counts().plot(kind="bar", ax=axes[0], color=["steelblue", "tomato"])
axes[0].set_title("Anomaly Class Distribution")
axes[0].set_xlabel("anomaly_present"); axes[0].set_ylabel("Number of records")
axes[0].tick_params(axis="x", rotation=0)

year.value_counts().sort_index().plot(kind="bar", ax=axes[1], color="slategray")
axes[1].set_title("Records by Year")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("Number of records")

rate_by_year = eda_target.groupby(year).mean() * 100
rate_by_year.plot(kind="bar", ax=axes[2], color="darkorange")
axes[2].set_title("Anomaly Rate by Year")
axes[2].set_xlabel("Year"); axes[2].set_ylabel("Anomaly rate (%)")
plt.tight_layout(); plt.show()

print(pd.DataFrame({"records": year.value_counts().sort_index(),
                    "anomaly_rate_%": rate_by_year.round(2)}))
print("\nNote: years with very few records give unreliable anomaly rates.")

In [ ]:
# One mean value per KPI per record (quick summary used for EDA only)
def kpi_mean(d, k):
    if isinstance(d, dict) and k in d:
        a = to_array(d[k])
        if a is not None:
            a = a[np.isfinite(a)]
            if a.size > 0:
                return a.mean()
    return np.nan

kpi_means = pd.DataFrame({k: df["KPIs_parsed"].apply(lambda d: kpi_mean(d, k)) for k in kpi_names})
plot_kpis = kpi_names[:6]
print("Plotting KPIs:", plot_kpis)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, k in zip(axes.ravel(), plot_kpis):
    sns.histplot(kpi_means[k].dropna(), bins=40, ax=ax, color="steelblue")
    ax.set_title(f"Distribution of mean {k}"); ax.set_xlabel(f"Mean {k}"); ax.set_ylabel("Count")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, k in zip(axes.ravel(), plot_kpis):
    plot_df = pd.DataFrame({"anomaly_present": df["anomaly_present"], k: kpi_means[k]}).dropna()
    sns.boxplot(data=plot_df, x="anomaly_present", y=k, order=["No", "Yes"], showfliers=False, ax=ax)
    ax.set_title(f"Mean {k}: Anomaly vs Normal"); ax.set_xlabel("anomaly_present"); ax.set_ylabel(f"Mean {k}")
plt.tight_layout(); plt.show()

In [ ]:
corr_cols = kpi_names[:12]
plt.figure(figsize=(9, 7))
sns.heatmap(kpi_means[corr_cols].corr(), annot=len(corr_cols) <= 8, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation of Mean KPI Values")
plt.tight_layout(); plt.show()

In [ ]:
cat_labels = ["zone", "application", "mobility", "congestion"]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, c in zip(axes.ravel(), cat_labels):
    rate = (eda_target.groupby(df[c]).mean() * 100).sort_values(ascending=False)
    rate.plot(kind="bar", ax=ax, color="tomato")
    ax.set_title(f"Anomaly Rate by {c}"); ax.set_xlabel(c); ax.set_ylabel("Anomaly rate (%)")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## 7. Feature Engineering
1. **KPI features:** for each KPI array we compute mean, standard deviation, minimum, maximum and median. Non-finite samples are ignored (counted in Section 5).
2. **Supplied statistics:** numeric values from `statistics_parsed` are flattened into columns. We then check whether they just repeat the KPI-derived features and drop the ones that do.
3. **Context features:** `zone`, `application`, `mobility`, `congestion` are included only if they pass a simple suitability check. `sampling_rate` is included if numeric.
4. **Temporal features are not used.** Section 9 shows whether the timestamps can be trusted. Hour/day features from unreliable timestamps would add noise.
5. Constant and near-duplicate features are removed (these checks do not use the target).

In [ ]:
STAT_NAMES = ["mean", "std", "min", "max", "median"]
rows = []
for d in df["KPIs_parsed"]:
    row = {}
    for k in kpi_names:
        a = to_array(d[k]) if isinstance(d, dict) and k in d else None
        if a is not None:
            a = a[np.isfinite(a)]
            if a.size == 0:
                a = None
        vals = [np.nan] * 5 if a is None else [a.mean(), a.std(), a.min(), a.max(), np.median(a)]
        for s, v in zip(STAT_NAMES, vals):
            row[f"{k}_{s}"] = v
    rows.append(row)
kpi_feats = pd.DataFrame(rows, index=df.index)
print("KPI-derived features:", kpi_feats.shape)
kpi_feats.head(3)

In [ ]:
def flatten_numeric(obj, prefix=""):
    """Collect scalar numeric values from a nested dict (arrays/text are skipped)."""
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            out.update(flatten_numeric(v, f"{prefix}_{k}" if prefix else str(k)))
    elif isinstance(obj, (int, float, np.integer, np.floating)) and not isinstance(obj, bool):
        out[prefix] = float(obj)
    return out

stats_feats = pd.DataFrame([flatten_numeric(d) for d in df["statistics_parsed"]], index=df.index)
stats_feats = stats_feats.add_prefix("stat_").replace([np.inf, -np.inf], np.nan)
print("Supplied numeric statistics:", stats_feats.shape)

if stats_feats.shape[1] > 0:
    corr = pd.concat([kpi_feats, stats_feats], axis=1).corr()
    max_corr = corr.loc[stats_feats.columns, kpi_feats.columns].abs().max(axis=1)
    print("Max |correlation| of each supplied statistic with our KPI features:")
    print(max_corr.describe())
    redundant = max_corr[max_corr > 0.95].index.tolist()
    print(f"\n{len(redundant)} of {stats_feats.shape[1]} supplied statistics are redundant (|r| > 0.95) and will be dropped.")
    stats_keep = stats_feats.drop(columns=redundant)
else:
    stats_keep = stats_feats
print("Supplied statistics kept:", stats_keep.shape[1])

In [ ]:
# Suitability check for categorical context labels
cat_candidates = ["zone", "application", "mobility", "congestion"]
cat_cols = []
for c in cat_candidates:
    n_unique = df[c].nunique(dropna=True)
    miss = df[c].isna().mean() * 100
    rates = eda_target.groupby(df[c]).mean() * 100
    print(f"{c}: {n_unique} categories, {miss:.1f}% missing, anomaly rate range {rates.min():.2f}%-{rates.max():.2f}%")
    if 2 <= n_unique <= 20:
        cat_cols.append(c)
    else:
        print(f"   -> skipped (needs 2 to 20 categories)")
print("Categorical features used:", cat_cols)

sampling_num = pd.to_numeric(df["sampling_rate"], errors="coerce")
use_sampling = sampling_num.notna().mean() >= 0.99
print("sampling_rate numeric and usable:", use_sampling)

In [ ]:
parts = [kpi_feats, stats_keep]
if use_sampling:
    parts.append(sampling_num.rename("sampling_rate_num").to_frame())
X_all = pd.concat(parts, axis=1)
num_cols = list(X_all.columns)

# infinite -> NaN (imputed later using TRAINING medians only)
X_all[num_cols] = X_all[num_cols].replace([np.inf, -np.inf], np.nan)

# remove constant / empty features
constant = [c for c in num_cols if X_all[c].nunique(dropna=True) <= 1]
print("Constant or empty features removed:", len(constant))
num_cols = [c for c in num_cols if c not in constant]

# remove near-duplicate features (|r| > 0.98), keep the first of each pair
corr_abs = X_all[num_cols].corr().abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
redundant_num = [c for c in upper.columns if (upper[c] > 0.98).any()]
print("Near-duplicate features removed:", len(redundant_num))
num_cols = [c for c in num_cols if c not in redundant_num]

# add categorical features (missing label becomes its own 'Unknown' category)
for c in cat_cols:
    X_all[c] = df[c].fillna("Unknown").astype(str)

feature_cols = num_cols + cat_cols
X_all = X_all[feature_cols]
print("Final feature count:", len(feature_cols), f"({len(num_cols)} numeric, {len(cat_cols)} categorical)")
print("Missing values left (will be imputed in the pipeline):", int(X_all[num_cols].isna().sum().sum()))
print("Shape:", X_all.shape)

## 8. Target Preparation and Leakage Prevention

**Target:** `anomaly_present`, encoded `No = 0`, `Yes = 1`.

**Excluded from model inputs:** `description`, `QnA`, `anomalies_parsed`, the full `labels` dictionary, and `anomaly_present` itself. These describe or contain the answer.

**Remaining caution.** KPI statistics are computed from the *same trace* whose anomaly label we are predicting. So this is **retrospective classification** ("is this recorded trace anomalous?"). It is **not** prospective outage prediction ("will the service fail soon?"). Section 7 also checked that the supplied `statistics` do not just duplicate the KPI features.

**Is there a real outage / availability label?** The next cell searches the column names, label keys and KPI names.

In [ ]:
y_all = df["anomaly_present"].map({"No": 0, "Yes": 1})
unexpected = df.loc[y_all.isna(), "anomaly_present"].unique()
print("Records with missing/unexpected target:", y_all.isna().sum(), unexpected)
valid = y_all.notna()                      # only rows with a known label are used for modelling
print("Records used for modelling:", valid.sum(), "of", len(df))

counts = y_all[valid].astype(int).value_counts().sort_index()
print("\nClass counts (0=No, 1=Yes):\n", counts)
print(f"Positive rate: {counts.get(1, 0) / counts.sum() * 100:.2f}%  -> imbalanced target")

# ---- leakage guard ----
forbidden = ["anomaly_present", "description", "QnA", "anomalies", "labels"]
leaks = [c for c in X_all.columns if any(c == f or c.startswith(f + "_") for f in forbidden)]
assert not leaks, f"Leakage risk, remove: {leaks}"
assert len(X_all) == len(y_all) and (X_all.index == y_all.index).all(), "Features and target are misaligned"
print("\nLeakage check passed. Features and target are aligned.")

# ---- is there an outage/availability target? ----
keywords = ["outage", "unavail", "availability", "downtime", "service_down", "failure"]
searched = list(df.columns) + list(first_valid("labels_parsed").keys()) + kpi_names
hits = [s for s in searched if any(w in str(s).lower() for w in keywords)]
print("\nOutage/availability-related fields found:", hits if hits else "NONE")
if not hits:
    print("Conclusion: no defensible outage label exists. We keep anomaly classification as the task.")

**What would be needed for genuine outage prediction:** confirmed outage/availability labels (start and end times of real incidents), continuous time series per service/cell with trustworthy timestamps, and a forecasting setup (use data up to time *t* to predict an outage in a future window *t+h*). None of this is available, so none is invented here.

## 9. Data Splitting
A chronological split only makes sense if timestamps are trustworthy. The cell below tests this using clear rules: no invalid dates, no end-before-start, and no dates in the future. The records spanning up to 2037 are suspicious because they lie in the future relative to when data could have been collected.

- **Reliable timestamps** → chronological 70/15/15 split.
- **Unreliable timestamps** → stratified 70/15/15 split, **disclosed as retrospective classification only**.

A random/stratified split does **not** demonstrate future outage prediction. Preprocessing (imputation, scaling, encoding) is fitted on the training set only, inside the pipeline.

In [ ]:
reasons = []
n_nat = df["start_dt"].isna().sum() + df["end_dt"].isna().sum()
n_neg = (df["duration_sec"] <= 0).sum()
n_future = (df["start_dt"] > pd.Timestamp.now(tz="UTC")).sum()
if n_nat > 0:    reasons.append(f"{n_nat} invalid timestamps")
if n_neg > 0:    reasons.append(f"{n_neg} records with end_time <= start_time")
if n_future > 0: reasons.append(f"{n_future} records dated in the future (up to {df['start_dt'].max().year})")
timestamps_reliable = len(reasons) == 0
reliability_reasons = reasons
print("Timestamps reliable:", timestamps_reliable)
print("Reasons for doubt:", reasons if reasons else "none")

model_df = X_all.loc[valid].copy()
model_df["target"] = y_all[valid].astype(int)
model_df["start_dt"] = df.loc[valid, "start_dt"]

def stratified_split(m):
    train, tmp = train_test_split(m, test_size=0.30, stratify=m["target"], random_state=RANDOM_STATE)
    val, test = train_test_split(tmp, test_size=0.50, stratify=tmp["target"], random_state=RANDOM_STATE)
    return train, val, test

split_method = None
if timestamps_reliable:
    m = model_df.sort_values("start_dt").reset_index(drop=True)
    n = len(m)
    train_df, val_df, test_df = m.iloc[:int(.70 * n)], m.iloc[int(.70 * n):int(.85 * n)], m.iloc[int(.85 * n):]
    if min(train_df["target"].sum(), val_df["target"].sum(), test_df["target"].sum()) > 0:
        split_method = "Chronological 70/15/15"
    else:
        print("A chronological split left a set with no anomalies -> using stratified split instead.")
if split_method is None:
    train_df, val_df, test_df = stratified_split(model_df)
    split_method = "Stratified 70/15/15 (retrospective classification only; timestamps unreliable)"

print("\nSplit method:", split_method)
X_train, y_train = train_df[feature_cols], train_df["target"]
X_val,   y_val   = val_df[feature_cols],   val_df["target"]
X_test,  y_test  = test_df[feature_cols],  test_df["target"]
print(pd.DataFrame({"records": [len(y_train), len(y_val), len(y_test)],
                    "anomalies": [y_train.sum(), y_val.sum(), y_test.sum()],
                    "anomaly_rate_%": [round(y.mean() * 100, 2) for y in (y_train, y_val, y_test)]},
                   index=["train", "validation", "test"]))

## 10. Model Training
Models: **Dummy** (always predicts the majority class, a baseline), **Logistic Regression** and **Random Forest**. Both real models use `class_weight` to handle the imbalance. No oversampling is used. Each model sits in a Pipeline: median imputation → (scaling for Logistic Regression) → one-hot encoding of categories → classifier. All of it is fitted on the training set only.

In [ ]:
def build_pipeline(model, scale):
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scaler", StandardScaler()))
    transformers = [("num", Pipeline(num_steps), num_cols)]
    if cat_cols:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols))
    return Pipeline([("prep", ColumnTransformer(transformers)), ("model", model)])

models = {
    "Dummy (baseline)": build_pipeline(DummyClassifier(strategy="most_frequent"), scale=False),
    "Logistic Regression": build_pipeline(
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE), scale=True),
    "Random Forest": build_pipeline(
        RandomForestClassifier(n_estimators=200, min_samples_leaf=2, class_weight="balanced_subsample",
                               n_jobs=-1, random_state=RANDOM_STATE), scale=False),
}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    print("Trained:", name)

## 11. Model Evaluation
Metrics (positive class = anomaly):
- **Accuracy** = (TP+TN) / all
- **Precision** = TP / (TP+FP)
- **Recall** = TP / (TP+FN)
- **F1-score** = harmonic mean of precision and recall
- **False alarm rate** = FP / (FP+TN)

**Why accuracy is not enough:** only about 3.86% of records are anomalies. A model that always says "No anomaly" is about 96% accurate yet detects nothing (recall = 0). That is exactly what the Dummy baseline does, so we focus on recall, precision, F1 and false alarm rate.

In [ ]:
def evaluate(pipe, X, y):
    pred = pipe.predict(X)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"Accuracy": accuracy_score(y, pred),
            "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0),
            "F1": f1_score(y, pred, zero_division=0),
            "False Alarm Rate": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
            "TP": tp, "FP": fp, "TN": tn, "FN": fn}

val_results = pd.DataFrame({n: evaluate(p, X_val, y_val) for n, p in models.items()}).T
test_results = pd.DataFrame({n: evaluate(p, X_test, y_test) for n, p in models.items()}).T

print("VALIDATION results"); display(val_results.round(4))
print("TEST results");       display(test_results.round(4))

In [ ]:
# Confusion matrices on the test set
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, pipe) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, pipe.predict(X_test), labels=[0, 1],
                                            display_labels=["No anomaly", "Anomaly"],
                                            cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(f"Confusion Matrix - {name}\n(Test set)")
    ax.grid(False)
plt.tight_layout(); plt.show()

for name, pipe in models.items():
    print(f"===== {name} (test) =====")
    print(classification_report(y_test, pipe.predict(X_test), target_names=["No anomaly", "Anomaly"], zero_division=0))

In [ ]:
# Precision-recall curves on the test set
plt.figure(figsize=(7, 5))
for name in ["Logistic Regression", "Random Forest"]:
    proba = models[name].predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, proba)
    plt.plot(rec, prec, label=f"{name} (AP = {average_precision_score(y_test, proba):.3f})")
plt.axhline(y_test.mean(), color="gray", linestyle="--", label=f"Baseline (positive rate = {y_test.mean():.3f})")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall Curve (Test set)")
plt.legend(); plt.show()

## 12. Model Comparison
The best model is chosen by **validation F1-score** (not accuracy, and not the test set). The false alarm rate is shown next to it so we can check that the choice is acceptable. The test set is only used to report the final performance.

In [ ]:
comparison = pd.concat({"Validation": val_results[["Accuracy", "Precision", "Recall", "F1", "False Alarm Rate"]],
                        "Test": test_results[["Accuracy", "Precision", "Recall", "F1", "False Alarm Rate"]]}, axis=1)
display(comparison.round(4))

candidates = val_results.drop(index="Dummy (baseline)")
best_name = candidates["F1"].astype(float).idxmax()
best_pipe = models[best_name]
print(f"Best model by validation F1: {best_name}")
print(f"  Validation F1 = {val_results.loc[best_name, 'F1']:.4f}, "
      f"Recall = {val_results.loc[best_name, 'Recall']:.4f}, "
      f"False alarm rate = {val_results.loc[best_name, 'False Alarm Rate']:.4f}")
print(f"  Test       F1 = {test_results.loc[best_name, 'F1']:.4f}, "
      f"Recall = {test_results.loc[best_name, 'Recall']:.4f}, "
      f"False alarm rate = {test_results.loc[best_name, 'False Alarm Rate']:.4f}")

## 13. Feature Importance and Interpretation
Random Forest importance shows which features the model used most to separate anomalous and normal traces.

**How to read it (cautions)**
- Importance shows *association*, **not causation**.
- KPI summary features describe the same trace that was labelled, so a high importance means "the trace statistics look different when an anomaly is present", not "this KPI causes outages".
- Correlated features (e.g. mean and median of the same KPI) share importance, which can hide or exaggerate individual features. We removed near-duplicates (|r| > 0.98), but milder redundancy remains.

In [ ]:
rf_pipe = models["Random Forest"]
names = [n.split("__", 1)[1] for n in rf_pipe.named_steps["prep"].get_feature_names_out()]
importances = pd.Series(rf_pipe.named_steps["model"].feature_importances_, index=names).sort_values(ascending=False)

top = importances.head(15).sort_values()
plt.figure(figsize=(8, 6))
plt.barh(top.index, top.values, color="seagreen")
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance (mean decrease in impurity)"); plt.ylabel("Feature")
plt.tight_layout(); plt.show()
print(importances.head(10).round(4))

## 14. Prediction Demonstration
We apply the best model to **held-out test records** and show the predicted anomaly probability, predicted class and actual class. For illustration we show 5 true anomalies and 5 normal records (otherwise a random sample would contain almost no anomalies).

> This is **retrospective classification** of already-recorded traces. It is **not** a forecast of future network failure.

In [ ]:
pos = test_df[test_df["target"] == 1]
neg = test_df[test_df["target"] == 0]
demo = pd.concat([pos.sample(min(5, len(pos)), random_state=RANDOM_STATE),
                  neg.sample(min(5, len(neg)), random_state=RANDOM_STATE)])

proba = best_pipe.predict_proba(demo[feature_cols])[:, 1]
pred = best_pipe.predict(demo[feature_cols])
label_text = {0: "No anomaly", 1: "Anomaly"}
demo_table = pd.DataFrame({
    "start_time": demo["start_dt"].values,
    "anomaly_probability": proba.round(4),
    "predicted_class": [label_text[p] for p in pred],
    "actual_class": [label_text[t] for t in demo["target"]],
}, index=demo.index)
demo_table["correct"] = demo_table["predicted_class"] == demo_table["actual_class"]
print(f"Model used: {best_name}")
display(demo_table)

def predict_record(test_row_index):
    """Predict one held-out record by its index in test_df."""
    row = test_df.loc[[test_row_index], feature_cols]
    p = best_pipe.predict_proba(row)[0, 1]
    return {"anomaly_probability": round(float(p), 4),
            "predicted": label_text[int(p >= 0.5)],
            "actual": label_text[int(test_df.loc[test_row_index, "target"])]}

print("Example single-record prediction:", predict_record(demo.index[0]))

## 15. Conclusion
**What was done.** The nested telecom dataset was parsed, validated and summarized into numeric KPI features. Three models (Dummy, Logistic Regression, Random Forest) were trained with leakage prevention and compared on a held-out test set with Accuracy, Precision, Recall, F1 and False Alarm Rate. The exact numbers are printed in the final report cell below, generated from this run.

**Key points**
- The implemented task is **anomaly classification**. The dataset has no outage/availability label, so genuine service-outage prediction was **not** performed and no labels were invented.
- Accuracy is misleading for a ~3.86% positive class. Recall, precision, F1 and false alarm rate give the real picture.
- Timestamps spanning to 2037 are suspicious, which affects the choice of split (see the report below).

**Limitations**
- Features come from the same trace as the label, so this is retrospective.
- Fixed 0.5 decision threshold, no hyperparameter tuning, and no cross-validation (kept simple on purpose).
- Anomaly labels may be partly synthetic or rule-based, which could make them easier or harder to learn than real faults.

**Future improvements.** Tune the decision threshold for an acceptable false alarm rate, try cross-validation and XGBoost, and add richer time-series features.

**For genuine outage prediction you would need:** confirmed outage labels with start/end times, continuous per-service time series with trustworthy timestamps, and a forecasting setup that predicts an outage in a future window using only past data.

In [ ]:
print("=" * 60)
print("FINAL REPORT (generated from this notebook run)")
print("=" * 60)
print(f"Records loaded: {len(df)} | used for modelling: {len(model_df)}")
print(f"Features used: {len(feature_cols)} ({len(num_cols)} numeric, {len(cat_cols)} categorical)")
print(f"Anomaly rate in modelling data: {model_df['target'].mean() * 100:.2f}%")
print(f"Timestamps reliable: {timestamps_reliable}" + (f"  (issues: {'; '.join(reliability_reasons)})" if reliability_reasons else ""))
print(f"Split method: {split_method}")
print(f"Outage/availability label found: {'YES: ' + str(hits) if hits else 'NO - task remains anomaly classification'}")
print("\nTest-set results:")
print(test_results[["Accuracy", "Precision", "Recall", "F1", "False Alarm Rate"]].round(4).to_string())
print(f"\nBest model (by validation F1): {best_name}")
print("Top 5 features (Random Forest):", list(importances.head(5).index))
print("\nReminder: this is retrospective anomaly classification, not future outage prediction.")